In [1]:
# !pip install -q rasterio

import os, random, warnings, gc
from pathlib import Path
from typing import Optional
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms.functional as TF
import torchvision.models as models
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    precision_recall_curve
)
import rasterio

warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

torch.backends.cudnn.benchmark = True

# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════
class CFG:
    ROOT             = Path('/kaggle/input/datasets/shreyansdeshpande/200x200/FINALCNNDATA_200')
    OUTPUT_DIR       = Path('/kaggle/working')

    TRAIN_SITE       = ROOT / 'train' / 'site'
    TRAIN_NOSITE     = ROOT / 'train' / 'no_site'
    VAL_SITE         = ROOT / 'validation' / 'site'
    VAL_NOSITE       = ROOT / 'validation' / 'no_site'
    TEST_SITE        = ROOT / 'test' / 'site'
    TEST_NOSITE      = ROOT / 'test' / 'no_site'

    IMG_SUFFIX       = '.tif'

    # 11-channel layout: 0-3 (Satellite: R, G, B, NIR), 4-10 (LiDAR derivatives)
    SAT_BANDS        = [0, 1, 2, 3]
    LIDAR_BANDS      = [4, 5, 6, 7, 8, 9, 10]
    IN_CHANNELS      = len(SAT_BANDS) + len(LIDAR_BANDS)  # Total 11 channels

    NUM_CLASSES      = 1
    IMG_SIZE         = 200

    BATCH_SIZE       = 32
    NUM_EPOCHS       = 30
    T_MAX            = NUM_EPOCHS
    ETA_MIN          = 1e-6
    BASE_LR          = 1e-4
    WEIGHT_DECAY     = 0.05
    DROPOUT          = 0.6
    SEED             = 42
    NUM_WORKERS      = 2
    USE_AMP          = torch.cuda.is_available()

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CFG.SEED)
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ══════════════════════════════════════════════════════════════════════════════
# DATA INGESTION & DECOUPLED NORMALIZATION PATHWAYS
# ══════════════════════════════════════════════════════════════════════════════
def folder_to_df(site_dir: Path, nosite_dir: Path) -> pd.DataFrame:
    rows = []
    for label, folder in [(1, site_dir), (0, nosite_dir)]:
        files = list(folder.glob(f'*{CFG.IMG_SUFFIX}'))
        for f in sorted(files):
            rows.append({'filepath': str(f), 'label': label})
    return pd.DataFrame(rows, columns=['filepath', 'label'])

print('Scanning dataset folders...')
train_df = folder_to_df(CFG.TRAIN_SITE, CFG.TRAIN_NOSITE)
val_df   = folder_to_df(CFG.VAL_SITE,   CFG.VAL_NOSITE)
test_df  = folder_to_df(CFG.TEST_SITE,  CFG.TEST_NOSITE)

print(f"  Train : {len(train_df):<5} (site: {(train_df['label']==1).sum()}, no_site: {(train_df['label']==0).sum()})")
print(f"  Val   : {len(val_df):<5} (site: {(val_df['label']==1).sum()}, no_site: {(val_df['label']==0).sum()})")
print(f"  Test  : {len(test_df):<5} (site: {(test_df['label']==1).sum()}, no_site: {(test_df['label']==0).sum()})")

def normalize_lidar(lidar_arr: np.ndarray) -> np.ndarray:
    """Per-channel z-score standardization for continuous terrain derivatives."""
    out = np.empty_like(lidar_arr, dtype=np.float32)
    for c in range(lidar_arr.shape[0]):
        ch = lidar_arr[c]
        mean = float(np.nanmean(ch))
        std = float(np.nanstd(ch))
        if std < 1e-4 or np.isnan(std):
            out[c] = ch - mean
        else:
            out[c] = (ch - mean) / (std + 1e-6)
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)

def normalize_satellite(sat_arr: np.ndarray) -> np.ndarray:
    """Fixed [0.0, 1.0] scaling matched to Satlas/Sentinel-2 pretraining."""
    sat_arr = np.nan_to_num(sat_arr, nan=0.0, posinf=0.0, neginf=0.0)
    sat_arr = np.clip(sat_arr, 0.0, None)
    max_val = float(np.max(sat_arr))

    if max_val > 255.0:
        refl = sat_arr / 10000.0
        tci_proxy = np.power(np.clip(refl * 2.5, 0.0, 1.0), 1.0 / 2.2)
        return tci_proxy.astype(np.float32)
    elif max_val > 1.0:
        return (sat_arr / 255.0).astype(np.float32)
    else:
        return sat_arr.astype(np.float32)

def load_multimodal_patch(tif_path: str) -> np.ndarray:
    with rasterio.open(tif_path) as src:
        img = src.read().astype(np.float32)
    img = np.nan_to_num(img, nan=0.0, posinf=0.0, neginf=0.0)
    img = np.where((img < -1e4) | (img > 1e4), 0.0, img)

    sat_np = normalize_satellite(img[CFG.SAT_BANDS])
    lidar_np = normalize_lidar(img[CFG.LIDAR_BANDS])
    
    # Concatenate satellite [4, H, W] and LiDAR [7, H, W] into an 11-channel array [11, H, W]
    return np.concatenate([sat_np, lidar_np], axis=0)

class MultiModalDataset(Dataset):
    def __init__(self, df: pd.DataFrame, split_name: str, is_train: bool = True):
        self.df = df.reset_index(drop=True)
        self.is_train = is_train

        print(f"Preloading and caching {len(self.df)} {split_name} multimodal patches into RAM...")
        self.cached_samples = []
        for _, row in self.df.iterrows():
            combined_np = load_multimodal_patch(row['filepath'])
            self.cached_samples.append((combined_np, float(row['label'])))

    def __len__(self):
        return len(self.cached_samples)

    def _moderate_augment(self, t):
        if random.random() > 0.5:
            t = TF.hflip(t)
        if random.random() > 0.5:
            t = TF.vflip(t)
        angle = random.choice([0, 90, 180, 270])
        if angle != 0:
            t = TF.rotate(t, angle)
        if random.random() > 0.5:
            t = t + torch.randn_like(t) * 0.03
        return t

    def __getitem__(self, idx):
        combined_np, label = self.cached_samples[idx]
        t_tensor = torch.from_numpy(combined_np)
        t_tensor = TF.resize(t_tensor, [CFG.IMG_SIZE, CFG.IMG_SIZE], interpolation=TF.InterpolationMode.BILINEAR, antialias=True)

        if self.is_train:
            t_tensor = self._moderate_augment(t_tensor)

        return t_tensor, torch.tensor(label, dtype=torch.float32)

# ══════════════════════════════════════════════════════════════════════════════
# DATALOADERS
# ══════════════════════════════════════════════════════════════════════════════
class_counts = train_df['label'].value_counts().to_dict()
class_weights = {0: 1.0 / class_counts[0], 1: 1.0 / class_counts[1]}
sample_weights = [class_weights[label] for label in train_df['label']]
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(train_df), replacement=True)

trn_loader  = DataLoader(MultiModalDataset(train_df, "Train", is_train=True),
                         batch_size=CFG.BATCH_SIZE, sampler=sampler,
                         num_workers=CFG.NUM_WORKERS, pin_memory=True, drop_last=True)

val_loader  = DataLoader(MultiModalDataset(val_df, "Val", is_train=False),
                         batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
                         num_workers=CFG.NUM_WORKERS, pin_memory=True)

test_loader = DataLoader(MultiModalDataset(test_df, "Test", is_train=False),
                         batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
                         num_workers=CFG.NUM_WORKERS, pin_memory=True)

# ══════════════════════════════════════════════════════════════════════════════
# CONVNEXT MODEL ADAPTED FOR 11-CHANNEL MULTIMODAL INPUT
# ══════════════════════════════════════════════════════════════════════════════
def create_convnext_model(in_channels: int = 11, num_classes: int = 1, dropout: float = 0.6):
    model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
    
    old_conv = model.features[0][0]
    new_conv = nn.Conv2d(
        in_channels=in_channels,
        out_channels=old_conv.out_channels,
        kernel_size=old_conv.kernel_size,
        stride=old_conv.stride,
        padding=old_conv.padding,
        bias=old_conv.bias is not None
    )
    
    with torch.no_grad():
        if old_conv.weight.shape[1] == 3 and in_channels > 3:
            new_conv.weight[:, :3, :, :] = old_conv.weight
            mean_weight = old_conv.weight.mean(dim=1)  # Shape: [96, 4, 4]
            for c in range(3, in_channels):
                new_conv.weight[:, c, :, :] = mean_weight
        else:
            nn.init.kaiming_normal_(new_conv.weight, nonlinearity='relu')
            
        if old_conv.bias is not None and new_conv.bias is not None:
            new_conv.bias.copy_(old_conv.bias)
            
    model.features[0][0] = new_conv
    
    in_features = model.classifier[2].in_features  # 768
    model.classifier = nn.Sequential(
        model.classifier[0],  # LayerNorm2d
        model.classifier[1],  # Flatten
        nn.Dropout(p=dropout),
        nn.Linear(in_features, num_classes)
    )
    
    return model

# ══════════════════════════════════════════════════════════════════════════════
# LOSS, OPTIMIZER & TRAINING FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════
model = create_convnext_model(in_channels=CFG.IN_CHANNELS, num_classes=CFG.NUM_CLASSES, dropout=CFG.DROPOUT).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=CFG.BASE_LR, weight_decay=CFG.WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.T_MAX, eta_min=CFG.ETA_MIN)
scaler    = torch.amp.GradScaler(device=DEVICE.type, enabled=CFG.USE_AMP)
criterion = nn.BCEWithLogitsLoss()

best_val_auc = 0.0
ckpt_path    = CFG.OUTPUT_DIR / 'convnext_tiny_multimodal_best.pth'

def train_epoch(model, loader, optimizer, scaler):
    model.train()
    total_loss = 0.0
    all_labels, all_preds = [], []

    for inputs, labels in loader:
        inputs = inputs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=DEVICE.type, enabled=CFG.USE_AMP):
            logits = model(inputs).squeeze(1)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * labels.size(0)
        preds = torch.sigmoid(logits).detach().cpu().numpy()
        all_preds.extend(np.nan_to_num(preds, nan=0.5))
        all_labels.extend(labels.cpu().numpy())

    auc = roc_auc_score(all_labels, all_preds) if len(set(all_labels)) > 1 else 0.0
    pr_auc = average_precision_score(all_labels, all_preds) if len(set(all_labels)) > 1 else 0.0
    bin_preds = [1 if p >= 0.5 else 0 for p in all_preds]
    prec = precision_score(all_labels, bin_preds, zero_division=0)
    rec  = recall_score(all_labels, bin_preds, zero_division=0)
    f1   = f1_score(all_labels, bin_preds, zero_division=0)
    acc  = accuracy_score(all_labels, bin_preds)

    return total_loss / len(loader.dataset), auc, pr_auc, prec, rec, f1, acc

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    all_labels, all_preds = [], []

    for inputs, labels in loader:
        inputs = inputs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.amp.autocast(device_type=DEVICE.type, enabled=CFG.USE_AMP):
            logits = model(inputs).squeeze(1)
            loss = criterion(logits, labels)

        total_loss += loss.item() * labels.size(0)
        preds = torch.sigmoid(logits).cpu().numpy()
        all_preds.extend(np.nan_to_num(preds, nan=0.5))
        all_labels.extend(labels.cpu().numpy())

    auc = roc_auc_score(all_labels, all_preds) if len(set(all_labels)) > 1 else 0.0
    pr_auc = average_precision_score(all_labels, all_preds) if len(set(all_labels)) > 1 else 0.0
    bin_preds = [1 if p >= 0.5 else 0 for p in all_preds]
    prec = precision_score(all_labels, bin_preds, zero_division=0)
    rec  = recall_score(all_labels, bin_preds, zero_division=0)
    f1   = f1_score(all_labels, bin_preds, zero_division=0)
    acc  = accuracy_score(all_labels, bin_preds)

    return total_loss / len(loader.dataset), auc, pr_auc, prec, rec, f1, acc, all_labels, all_preds

# ══════════════════════════════════════════════════════════════════════════════
# TRAINING LOOP
# ══════════════════════════════════════════════════════════════════════════════
print(f'\n{"="*75}')
print(f'  Training ConvNeXt-Tiny on 11-Channel Multimodal Data ({CFG.NUM_EPOCHS} Epochs)')
print(f'{"="*75}')

for epoch in range(1, CFG.NUM_EPOCHS + 1):
    trn_loss, trn_auc, trn_pr_auc, trn_prec, trn_rec, trn_f1, trn_acc = train_epoch(model, trn_loader, optimizer, scaler)
    val_loss, val_auc, val_pr_auc, val_prec, val_rec, val_f1, val_acc, _, _ = evaluate(model, val_loader)

    scheduler.step()

    saved = ''
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(model.state_dict(), ckpt_path)
        saved = '  ✓ saved'

    print(f'Ep {epoch:02d}/{CFG.NUM_EPOCHS} | LR: {optimizer.param_groups[0]["lr"]:.2e}{saved}')
    print(f'  Train -> Loss: {trn_loss:.4f} | ROC-AUC: {trn_auc:.4f} | PR-AUC: {trn_pr_auc:.4f} | F1: {trn_f1:.4f}')
    print(f'  Val   -> Loss: {val_loss:.4f} | ROC-AUC: {val_auc:.4f} | PR-AUC: {val_pr_auc:.4f} | F1: {val_f1:.4f}')

# ══════════════════════════════════════════════════════════════════════════════
# TEST BENCHMARK & EVALUATION
# ══════════════════════════════════════════════════════════════════════════════
if ckpt_path.exists():
    print(f"\nLoading best checkpoint: {ckpt_path.name}")
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval()

    # Threshold calibration on Validation set
    _, _, _, _, _, _, _, val_labels, val_preds = evaluate(model, val_loader)
    precisions, recalls, thresholds = precision_recall_curve(val_labels, val_preds)
    f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-8)
    best_idx = np.argmax(f1_scores)
    best_thresh = float(thresholds[best_idx])

    # Final evaluation on untouched Test set
    _, tst_auc, tst_pr_auc, _, _, _, _, test_labels, test_preds = evaluate(model, test_loader)
    test_preds_np  = np.array(test_preds)
    test_labels_np = np.array(test_labels)

    bin_default = (test_preds_np >= 0.5).astype(int)
    bin_opt     = (test_preds_np >= best_thresh).astype(int)

    print("\n" + "=" * 65)
    print("      FINAL BENCHMARK: UNTOUCHED TEST SET (CONVNEXT-TINY MULTIMODAL)")
    print("=" * 65)
    print(f"Optimal Threshold Calibrated on Val : {best_thresh:.4f}")
    print(f"\nTest Metrics @ Default Threshold (0.50):")
    print(f"  • ROC-AUC     : {tst_auc:.4f}")
    print(f"  • PR-AUC      : {tst_pr_auc:.4f}")
    print(f"  • F1-Score    : {f1_score(test_labels_np, bin_default, zero_division=0):.4f}")
    print(f"  • Precision   : {precision_score(test_labels_np, bin_default, zero_division=0):.4f}")
    print(f"  • Recall      : {recall_score(test_labels_np, bin_default, zero_division=0):.4f}")
    print(f"  • Accuracy    : {accuracy_score(test_labels_np, bin_default):.4f}")

    print(f"\nTest Metrics @ Optimal Calibrated Threshold ({best_thresh:.4f}):")
    print(f"  • ROC-AUC     : {tst_auc:.4f}")
    print(f"  • PR-AUC      : {tst_pr_auc:.4f}")
    print(f"  • F1-Score    : {f1_score(test_labels_np, bin_opt, zero_division=0):.4f}")
    print(f"  • Precision   : {precision_score(test_labels_np, bin_opt, zero_division=0):.4f}")
    print(f"  • Recall      : {recall_score(test_labels_np, bin_opt, zero_division=0):.4f}")
    print(f"  • Accuracy    : {accuracy_score(test_labels_np, bin_opt):.4f}")
    print("=" * 65)

gc.collect()
torch.cuda.empty_cache()

Using device: cuda
Scanning dataset folders...
  Train : 3596  (site: 876, no_site: 2720)
  Val   : 1245  (site: 326, no_site: 919)
  Test  : 878   (site: 226, no_site: 652)
Preloading and caching 3596 Train multimodal patches into RAM...
Preloading and caching 1245 Val multimodal patches into RAM...
Preloading and caching 878 Test multimodal patches into RAM...
Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 179MB/s] 



  Training ConvNeXt-Tiny on 11-Channel Multimodal Data (30 Epochs)
Ep 01/30 | LR: 9.97e-05  ✓ saved
  Train -> Loss: 0.6853 | ROC-AUC: 0.5917 | PR-AUC: 0.5628 | F1: 0.5348
  Val   -> Loss: 0.6316 | ROC-AUC: 0.6935 | PR-AUC: 0.4770 | F1: 0.4789
Ep 02/30 | LR: 9.89e-05  ✓ saved
  Train -> Loss: 0.6276 | ROC-AUC: 0.6985 | PR-AUC: 0.6841 | F1: 0.6262
  Val   -> Loss: 0.5826 | ROC-AUC: 0.7219 | PR-AUC: 0.5534 | F1: 0.5062
Ep 03/30 | LR: 9.76e-05  ✓ saved
  Train -> Loss: 0.5797 | ROC-AUC: 0.7510 | PR-AUC: 0.7708 | F1: 0.6511
  Val   -> Loss: 0.6329 | ROC-AUC: 0.7229 | PR-AUC: 0.5784 | F1: 0.5071
Ep 04/30 | LR: 9.57e-05  ✓ saved
  Train -> Loss: 0.5577 | ROC-AUC: 0.7752 | PR-AUC: 0.7911 | F1: 0.6802
  Val   -> Loss: 0.5265 | ROC-AUC: 0.7353 | PR-AUC: 0.5849 | F1: 0.5212
Ep 05/30 | LR: 9.34e-05  ✓ saved
  Train -> Loss: 0.5393 | ROC-AUC: 0.7942 | PR-AUC: 0.8179 | F1: 0.7143
  Val   -> Loss: 0.6023 | ROC-AUC: 0.7409 | PR-AUC: 0.5996 | F1: 0.5116
Ep 06/30 | LR: 9.05e-05
  Train -> Loss: 0.5269